<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/L4S_06_curvas_entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L4S_06 — Curvas de Entrenamiento · 5-Fold CV

**Objetivo:** Analizar la dinámica de entrenamiento de los modelos DL:  
convergencia, overfitting, y efectividad del early stopping.

| Modelo | Historia disponible |
|--------|-------------------|
| U-Net ResNet-34 | ✅ Completa (loss + F1 + Dice + IoU por época, por fold) |
| ResNet-50 | ⚡ Parcial (best_epoch + F1 final por fold) |
| EfficientNet-B4 | ⚡ Parcial (best_epoch + F1 final por fold) |

> **Nota:** La historia completa de ResNet-50 y EfficientNet no fue guardada en JSON.  
> Esta celda incluye el código para habilitarla en futuros entrenamientos.


In [ ]:
# ── Celda 0: Entorno y Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

DRIVE_PATH = '/content/drive/MyDrive/Landslide4Sense'
ROOT = Path(DRIVE_PATH)
OUT_DIR = ROOT / 'results' / 'comparable_literatura' / 'curvas_entrenamiento'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'✅ Salida: {OUT_DIR}')


## 1. U-Net — Curvas completas por fold

In [ ]:
# ── Celda 1: Cargar historiales U-Net ────────────────────────────────────────
unet_folds = []
for k in range(1, 6):
    p = ROOT / f'results/comparable_literatura/unet_5fold/fold{k}_results.json'
    if p.exists():
        with open(p) as f:
            unet_folds.append(json.load(f))
    else:
        print(f'⚠️  No encontrado: {p}')

print(f'✅ {len(unet_folds)} folds U-Net cargados')
for r in unet_folds:
    h = r['history']
    best_ep = r['best_epoch']
    n_ep = len(h['train_loss'])
    early = n_ep < 20
    print(f'  Fold {r["fold"]}: {n_ep} épocas | best_epoch={best_ep} | '
          f'{"Early stop" if early else "Completó 20 épocas"} | '
          f'F1_px={r["f1_pixel_thr05"]:.4f}  Dice={r["dice_thr05"]:.4f}')


In [ ]:
# ── Celda 2: Gráfica train/val loss y F1 por fold ────────────────────────────
n_folds = len(unet_folds)
fig = plt.figure(figsize=(16, 4 * n_folds))
gs  = gridspec.GridSpec(n_folds, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORS = {'train': '#2E5FA3', 'val': '#EF4444', 'f1': '#16A34A',
          'dice': '#D97706', 'iou': '#9333EA'}

for row, r in enumerate(unet_folds):
    h      = r['history']
    epochs = range(1, len(h['train_loss']) + 1)
    best   = r['best_epoch']

    # Panel 1: Loss
    ax = fig.add_subplot(gs[row, 0])
    ax.plot(epochs, h['train_loss'], color=COLORS['train'], lw=2, label='Train loss')
    ax.plot(epochs, h['val_loss'],   color=COLORS['val'],   lw=2, label='Val loss',  ls='--')
    ax.axvline(best, color='black', lw=1, ls=':', alpha=0.7, label=f'Best ep={best}')
    ax.set_title(f'Fold {r["fold"]} — Loss', fontsize=10, fontweight='bold')
    ax.set_xlabel('Época'); ax.set_ylabel('Loss')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

    # Panel 2: F1 píxel
    ax = fig.add_subplot(gs[row, 1])
    ax.plot(epochs, h['val_f1_pixel'], color=COLORS['f1'],   lw=2, label='Val F1 pixel')
    ax.plot(epochs, h['val_dice'],     color=COLORS['dice'],  lw=2, label='Val Dice', ls='--')
    ax.axvline(best, color='black', lw=1, ls=':', alpha=0.7)
    ax.axhline(r['f1_pixel_thr05'], color=COLORS['f1'], lw=1, ls=':', alpha=0.5)
    ax.set_title(f'Fold {r["fold"]} — F1 pixel & Dice', fontsize=10, fontweight='bold')
    ax.set_xlabel('Época'); ax.set_ylabel('Métrica')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)
    ax.set_ylim(0, 1)

    # Panel 3: IoU + gap overfitting
    ax = fig.add_subplot(gs[row, 2])
    gap = [vl - tl for tl, vl in zip(h['train_loss'], h['val_loss'])]
    ax.plot(epochs, gap,            color='#F59E0B', lw=2, label='Val−Train loss (gap)')
    ax.plot(epochs, h['val_iou'],   color=COLORS['iou'], lw=2, label='Val IoU', ls='--')
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.axvline(best, color='black', lw=1, ls=':', alpha=0.7)
    ax.set_title(f'Fold {r["fold"]} — IoU & Gap overfitting', fontsize=10, fontweight='bold')
    ax.set_xlabel('Época'); ax.set_ylabel('Valor')
    ax.legend(fontsize=8); ax.grid(alpha=0.25)

plt.suptitle('U-Net ResNet-34 — Curvas de Entrenamiento (5 Folds)',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(OUT_DIR / 'unet_curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: unet_curvas_entrenamiento.png')


## 2. U-Net — Análisis de overfitting y convergencia

In [ ]:
# ── Celda 3: Análisis overfitting ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Gap máximo por fold
max_gaps  = []
best_eps  = []
final_f1s = []
for r in unet_folds:
    h = r['history']
    gap = [abs(vl - tl) for tl, vl in zip(h['train_loss'], h['val_loss'])]
    max_gaps.append(max(gap))
    best_eps.append(r['best_epoch'])
    final_f1s.append(r['f1_pixel_thr05'])

fold_nums = [r['fold'] for r in unet_folds]

# Panel 1: Gap val-train loss por fold
ax = axes[0]
ax.bar(fold_nums, max_gaps, color='#F59E0B', edgecolor='white', width=0.6)
ax.set_xlabel('Fold'); ax.set_ylabel('Max |Val−Train| loss')
ax.set_title('Gap máximo overfitting\n(menor = mejor generalización)',
             fontsize=10, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.axhline(np.mean(max_gaps), color='red', lw=1.5, ls='--', label=f'Media={np.mean(max_gaps):.3f}')
ax.legend(fontsize=9)
for i, (f, v) in enumerate(zip(fold_nums, max_gaps)):
    ax.text(f, v + 0.001, f'{v:.3f}', ha='center', fontsize=9)

# Panel 2: Best epoch por fold
ax = axes[1]
ax.bar(fold_nums, best_eps, color='#2E5FA3', edgecolor='white', width=0.6)
ax.set_xlabel('Fold'); ax.set_ylabel('Época del mejor checkpoint')
ax.set_title('Early stopping — época óptima\n(max=20)',
             fontsize=10, fontweight='bold')
ax.axhline(20, color='gray', lw=1, ls='--', label='Máx. épocas')
ax.axhline(np.mean(best_eps), color='red', lw=1.5, ls='--',
           label=f'Media={np.mean(best_eps):.1f}')
ax.set_ylim(0, 23); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
for i, (f, v) in enumerate(zip(fold_nums, best_eps)):
    ax.text(f, v + 0.3, str(v), ha='center', fontsize=9)

# Panel 3: F1 final por fold
ax = axes[2]
ax.bar(fold_nums, final_f1s, color='#16A34A', edgecolor='white', width=0.6)
ax.axhline(np.mean(final_f1s), color='red', lw=1.5, ls='--',
           label=f'Media={np.mean(final_f1s):.4f}')
ax.set_xlabel('Fold'); ax.set_ylabel('F1 pixel @ thr=0.5')
ax.set_title('F1 pixel por fold\n(desde mejor checkpoint)',
             fontsize=10, fontweight='bold')
ax.set_ylim(0, 1); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
for i, (f, v) in enumerate(zip(fold_nums, final_f1s)):
    ax.text(f, v + 0.01, f'{v:.4f}', ha='center', fontsize=8.5)

plt.tight_layout()
plt.savefig(OUT_DIR / 'unet_analisis_overfitting.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: unet_analisis_overfitting.png')


## 3. ResNet-50 y EfficientNet — Información disponible

In [ ]:
# ── Celda 4: ResNet-50 y EfficientNet — best_epoch y F1 por fold ─────────────
import json

def load_json(rel):
    p = ROOT / rel
    return json.load(open(p)) if p.exists() else None

_rn  = load_json('results/comparable_literature/resnet50_5fold/kfold5_summary.json')
_eff = load_json('results/comparable_literatura/efficientnet_5fold/kfold5_summary.json')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, data, nombre, color in [
    (axes[0], _rn,  'ResNet-50',      '#818CF8'),
    (axes[1], _eff, 'EfficientNet-B4','#A78BFA'),
]:
    if data is None:
        ax.text(0.5, 0.5, 'JSON no encontrado', ha='center', va='center',
                transform=ax.transAxes, fontsize=12)
        ax.set_title(nombre); continue

    folds = data['folds']
    fold_nums_m = [f['fold'] for f in folds]
    best_eps_m  = [f.get('best_epoch', 0) for f in folds]
    f1s_m       = [f['f1_thr05'] for f in folds]

    x = np.arange(len(fold_nums_m))
    w = 0.35
    b1 = ax.bar(x - w/2, best_eps_m, w, color=color, alpha=0.8,
                edgecolor='white', label='Best epoch')
    ax2 = ax.twinx()
    ax2.plot(x, f1s_m, 'o-', color='#EF4444', lw=2, ms=8, label='F1 patch')
    ax2.set_ylabel('F1-Score', fontsize=10, color='#EF4444')
    ax2.tick_params(axis='y', colors='#EF4444')
    ax2.set_ylim(0, 1)

    ax.set_xticks(x); ax.set_xticklabels([f'F{f}' for f in fold_nums_m])
    ax.set_xlabel('Fold'); ax.set_ylabel('Época del mejor checkpoint')
    ax.set_title(f'{nombre} — Best epoch & F1 por fold\n'
                 f'(media F1={np.mean(f1s_m):.4f} | media best_ep={np.mean(best_eps_m):.1f})',
                 fontsize=10, fontweight='bold')
    ax.axhline(20, color='gray', lw=1, ls='--', alpha=0.5, label='Max épocas')
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labs1 + labs2, fontsize=9, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 25)

plt.suptitle('ResNet-50 y EfficientNet-B4 — Épocas óptimas (5 Folds)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'dl_best_epochs.png', dpi=150, bbox_inches='tight')
plt.show(); print('Guardado: dl_best_epochs.png')


## 4. Cómo habilitar el historial completo en futuros entrenamientos

Para guardar el historial de ResNet-50 y EfficientNet en la próxima sesión,  
añade estas líneas en la celda de guardado JSON de L4S_02 y L4S_03  
(justo antes de `json.dump`):

```python
# Guardar también JSON por fold (incluye history)
for r in fold_results:
    fold_path = output_dir / f'fold{r["fold"]}_results.json'
    with open(fold_path, 'w') as f:
        json.dump(r, f, indent=2, default=str)
    print(f'  → fold{r["fold"]}_results.json guardado')
```

Esto generará archivos `fold{N}_results.json` con el campo `history` completo,  
exactamente igual que L4S_04 (U-Net), y este notebook los leerá automáticamente.


## 5. Resumen

In [ ]:
# ── Celda 5: Resumen ──────────────────────────────────────────────────────────
print('='*60)
print('  RESUMEN — CURVAS DE ENTRENAMIENTO')
print('='*60)
if unet_folds:
    all_best_eps = [r['best_epoch'] for r in unet_folds]
    all_f1s      = [r['f1_pixel_thr05'] for r in unet_folds]
    all_dice     = [r['dice_thr05']     for r in unet_folds]
    print(f'\n  U-Net ResNet-34 (5 folds):')
    print(f'    Best epoch promedio : {np.mean(all_best_eps):.1f} / 20')
    print(f'    Early stop activado : {sum(e < 20 for e in all_best_eps)}/5 folds')
    print(f'    F1 pixel media      : {np.mean(all_f1s):.4f} ± {np.std(all_f1s):.4f}')
    print(f'    Dice media          : {np.mean(all_dice):.4f} ± {np.std(all_dice):.4f}')
    gap_vals = []
    for r in unet_folds:
        h = r['history']
        gap = [abs(vl - tl) for tl, vl in zip(h['train_loss'], h['val_loss'])]
        gap_vals.append(max(gap))
    print(f'    Gap overfitting max : {np.mean(gap_vals):.4f} ± {np.std(gap_vals):.4f}')
print(f'\n  Figuras guardadas en: {OUT_DIR}')
print('='*60)
